## Exercise 9 Task 1
### Register model, deploy as web service test the web service

Deploy your trained model as a web service and test your deployment.

Import necessary packages.

In [ ]:
#Ensure you are using Python 3.6. To switch Python versions, select Kernel >> Change kernel
import azureml.core
import pandas as pd
from azureml.core.workspace import Workspace
import logging
import os

Check the version of Azure ML

In [ ]:
#Check the version of Azure ML
print("Azure ML SDK Version: ", azureml.core.VERSION)

Connect to the workspace you created for this lab.

In [ ]:
ws = Workspace.from_config()

View workspae properties to make sure that you are using the correct workspace.

In [ ]:
#To view the details of your workspace
ws.get_details()

The first step is to register the model that you want to deploy.

First run the code in the next cell to view a list of models registered in the workspace.

In [ ]:
#List models registered in the workspace

from azureml.core.model import Model

model_list = Model.list(workspace=ws)
for i in model_list:
  print("Model Name:", i.name,"\tVersion:", i.version, "\tDescription:", 
  i.description, i.tags)

If the code in the previous cell did not return any models, or if the model that you trained and want to deploy is not registered in the workspace, run the code in the next cell to register it. To register a model, you will need the full path to its pkl file.  

In [ ]:
from azureml.core.model import Model

model = Model.register(model_path = "credit_default.pkl",
                      model_name = "credit-default-model",
                      tags = {'area' : "cost", 'type' : "classification"},
                      description = "Classifies people with high default probability",
                      workspace = ws)

List models in the workspace

In [ ]:
model_list = Model.list(workspace=ws)
for i in model_list:
  print("Model Name:", i.name,"\tVersion:", i.version, "\tDescription:", 
  i.description, i.tags)

Create a container image.

A container image requires a scoring file, so we need to first create a scoring file.

The scoring file is used by the web service to show how to use the model.

In [ ]:
%%writefile score.py

import json
import numpy as np
import os
import pickle
#from sklearn.externals import joblib
import joblib
from azureml.core.model import Model
from sklearn.linear_model import ElasticNet

def init():
    global model
    model_path = Model.get_model_path('credit-default-model')
    model = joblib.load(model_path)

def run(raw_data):
    try:
        data = np.array(json.loads(raw_data)['data'])
        result = model.predict(data)
        # Any data type can be returned if json can be serialized
        return result.tolist()
    except Exception as e:
       error = str(e)
       return error

Check the project root folder and verify that the scoring file score.py was created.

Next we create an environment configuration (yml) file.

In [ ]:
from azureml.core.conda_dependencies import CondaDependencies 

MyModelEnv = CondaDependencies.create(conda_packages=['scikit-learn'])
with open("credit_default_model_env.yml","w") as f:
    f.write(MyModelEnv.serialize_to_string())

Now we are ready to deploy the model. We are going to deploy the model to an Azure Container Instance (ACI).

First we need to specify the configuration of the ACI we want to use for our deployment.

In [ ]:
from azureml.core.webservice import AciWebservice

aciconfig = AciWebservice.deploy_configuration(cpu_cores = 2, 
                                              memory_gb = 2, 
                                              tags = {"data": "credit default", "type": "classification"}, 
                                               description = 'Classifies people with high default probability')

Now we will deploy in the ACI container

In [ ]:
%%time

from azureml.core.webservice import Webservice
from azureml.core.model import InferenceConfig

inference_config = InferenceConfig(runtime= "python", 
                                   entry_script="entry_script.py",
                                   conda_file="credit_default_model_env.yml")

service = model.deploy(workspace=ws, 
                       name='credit-default-service', 
                       models=[model], 
                       inference_config=inference_config, 
                       deployment_config=aciconfig)

service.wait_for_deployment(show_output=True)

To send data to the deployed web service and get the prediction, you will need the web service URL.

The code in the next cell displays the web service URL.

In [ ]:
print(service.scoring_uri)

Test the web service by sending sample data.

Use different values of input data and compare the predictions. Is there any feature that does not impact the prediction?

In [ ]:
import json

#input data from left to right: vendor, pickup_weekday, pickup_hour, passengers, distance
test_input = json.dumps({'data': [[20000,2,2,1,24,2,2,0,0,0,0,3913,3102,689,0,0,0,0,689,0,0,0,0]]})

prediction = service.run(test_input)

print(prediction)

You can also test the web service using the following method.

In [ ]:
import requests

#input data from left to right: vendor, pickup_weekday, pickup_hour, passengers, distance
test_input2 = json.dumps({'data': [[20000,2,2,1,24,2,2,0,0,0,0,3913,3102,689,0,0,0,0,689,0,0,0,0]]})

headers = {'Content-Type':'application/json'}
resp = requests.post(service.scoring_uri, test_input2, headers=headers)

prediction = json.loads(resp.text)
print(prediction)
